<img src="https://raw.githubusercontent.com/steinflow47/stein-flow-trainer/main/images/train_classify.png" width="64" align="left" alt="">

**Stein Flow — Train Classify** · train a classifier on a dataset of class folders from Stein Flow (Train Classify → More → Export for notebook), then download the model for Classify.

Pressing ▶ means you accept the [terms of use](https://github.com/steinflow47/stein-flow-trainer/blob/main/TERMS.txt). · Flint


In [ ]:
#@title Stein Flow — Train Classify { display-mode: "form" }
#@markdown Press ▶ to start. The first run installs the trainer and NVIDIA's CUDA 13 runtime (a few minutes).
import os, subprocess, sys, threading, time
from IPython.display import HTML, display

REPO = "steinflow47/stein-flow-trainer"
B = "/content/stein-trainer/stein_trainer"

def _gpu():
    """The runtime's GPU, as nvidia-smi names it; "" when it has none."""
    try:
        out = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30).stdout
    except (OSError, subprocess.TimeoutExpired):
        return ""
    return out.strip().splitlines()[0] if "GPU" in out else ""

NO_GPU = ('<div style="border-left:4px solid #d1453b;padding:10px 14px;border-radius:6px;'
          'background:rgba(209,69,59,.10);font-size:14px;line-height:1.5">'
          '<b>This runtime has no GPU</b>, and training on the CPU would take days.<br>'
          'Choose <b>Runtime → Change runtime type → T4 GPU</b>, save, then press ▶ again. '
          'Nothing was installed.</div>')

def _ready():
    status = display(HTML(""), display_id=True)
    p = subprocess.Popen(f"curl -fsSL https://raw.githubusercontent.com/{REPO}/main/bootstrap.sh | bash",
                         shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, step = [], ["Starting"]
    def read():
        for line in p.stdout:
            lines.append(line)
            if line.startswith("• "):
                step[0] = line[2:].strip()
    reader = threading.Thread(target=read, daemon=True)
    reader.start()
    t0 = time.time()
    while reader.is_alive():
        status.update(HTML(f'<span style="opacity:.6">⏳ {step[0]} … {int(time.time() - t0)} s</span>'))
        time.sleep(0.5)
    p.wait()
    status.update(HTML(""))
    if p.returncode != 0:
        print("".join(lines))
        print("Setup failed - the lines above say why.")
        return False
    os.environ["STEIN_TRAIN_EXE"] = f"{B}/bin/stein_train"
    if f"{B}/python" not in sys.path:
        sys.path.insert(0, f"{B}/python")
    t = subprocess.run([f"{B}/bin/stein_train", "--describe"], capture_output=True, text=True)
    if t.returncode != 0:
        missing = subprocess.run(f"ldd {B}/bin/stein_train | grep 'not found' | sort -u",
                                 shell=True, capture_output=True, text=True).stdout.strip()
        print("The trainer cannot start on this runtime.")
        print(t.stderr.strip()[-400:])
        if missing:
            print("Missing:\n" + missing)
        return False
    return True

if not _gpu():
    display(HTML(NO_GPU))
elif _ready():
    from stein_notebook import Workbench
    display(Workbench(task="classification"))